# BioEmu-2 · self-contained GPU readout adaptation
Public mdCATH → pinned BioEmu-2 → 20 training steps → held-out structures, loss and GPU readings. All recipe code is in these cells. No external helper scripts or platform SDK are required.

Experimental: 1,152 readout weights are updated; the rest stays frozen. Two proteins, fixed-noise denoising, unknown pretraining overlap. Use a dedicated Python 3.12 GPU kernel. Optional HTTP reporting and W&B credentials come only from environment variables.

In [ ]:
import sys, subprocess, importlib.util
if importlib.util.find_spec('pip') is None:
    subprocess.check_call([sys.executable, '-m', 'ensurepip'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + ['--index-url', 'https://packagefeedproxy.microsoft.io/pypi/simple/', 'bioemu2[cuda,viz] @ https://github.com/microsoft/bioemu2/archive/06be369dd555849165a737472d42d936959e46fc.tar.gz', 'h5py', 'matplotlib'])

# Tracking packages come from PyPI; the Microsoft feed hosts the model dependencies.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--index-url', 'https://pypi.org/simple', 'wandb==0.31.0'])


## Download and verify public mdCATH data

In [ ]:
from pathlib import Path
import hashlib,json,urllib.request
root=Path.cwd()/'bioemu2'/'data';root.mkdir(parents=True,exist_ok=True);manifest=[]
rev='5e3ed8aec62b689e01751db16275fdcdbc39e47f'
expected={'1ha8A00':'e06d8cdb5d8b6fef7e1a5c23df7b95c77972bc6c90c5cdbe7e17404291bcc8cc','1e8rA00':'714a5382ca040a23988a952a0fda0951eb8db4c64110e879bfe5420f43c73a4a'}
for domain in expected:
 p=root/f'mdcath_dataset_{domain}.h5'
 url=f'https://huggingface.co/datasets/compsciencelab/mdCATH/resolve/{rev}/data/{p.name}'
 if not p.exists():
  temporary=p.with_suffix('.download');urllib.request.urlretrieve(url,temporary);temporary.replace(p)
 digest=hashlib.sha256(p.read_bytes()).hexdigest()
 if digest != expected[domain]:raise ValueError(f'Checksum mismatch: {domain}')
 manifest.append({'domain':domain,'file':str(p),'url':url,'revision':rev,'sha256':hashlib.sha256(p.read_bytes()).hexdigest(),'bytes':p.stat().st_size})
(root/'manifest.json').write_text(json.dumps(manifest,indent=2))

In [ ]:
from pathlib import Path
"""Optional W&B tracking; credentials are read from the environment."""
import os

### Start run

In [ ]:
def start_run(run_id, config, api):
    import wandb
    run=wandb.init(project=os.environ.get('WANDB_PROJECT','open-chem-recipes'),
        entity=os.environ.get('WANDB_ENTITY'),id=run_id,name=run_id,config=config,
        mode='online' if os.environ.get('WANDB_API_KEY') else 'offline',
        dir=str(Path.cwd()),tags=['cookbook'])
    print('W&B:',run.url if os.environ.get('WANDB_API_KEY') else 'offline local tracking')
    return run


### Finish run

In [ ]:
def finish_run(run, metrics):
    run.summary.update({k:v for k,v in metrics.items() if isinstance(v,(int,float,str,bool))})
    for sample in metrics.get('gpu_samples',[]):
        run.log({'gpu/'+k:v for k,v in sample.items() if isinstance(v,(int,float))})
    run.finish()

In [ ]:
"""Experimental BioEmu-2 readout adaptation; not an official Microsoft training recipe."""
import argparse
import base64
import hashlib
import io
import json
import os
from pathlib import Path
import time
import subprocess
import threading
from datetime import datetime, timezone
import urllib.request

ROOT = Path.cwd()/'bioemu2'
ROOT.mkdir(parents=True, exist_ok=True)
SOURCE_REV = '06be369dd555849165a737472d42d936959e46fc'
MODEL_REV = 'dd4b0c1b69984b68edcb6200a544213be4d75242'
PARAMETER = 'score_model.structure_module.readout.to_out.1.weight'
DOMAINS = ('1ha8A00', '1e8rA00')

### Local run records

In [ ]:
def api(method, path, body=None, raw=False):
    """Local JSON run records and content-addressed artifacts; no server required."""
    import uuid
    store=Path.cwd()/'results'/'records';store.mkdir(parents=True,exist_ok=True)
    if method=='POST' and path=='/artifacts':
        data=base64.b64decode(body['content_b64']);digest=hashlib.sha256(data).hexdigest()
        (store/digest).write_bytes(data)
        return {'artifact_id':digest}
    if method=='POST' and path=='/training/runs':
        key='train_'+uuid.uuid4().hex[:16]
        (store/(key+'.json')).write_text(json.dumps(body,indent=2))
        return {'training_run_id':key}
    if method=='PATCH':
        target=store/(path.rsplit('/',1)[-1]+'.json')
        prior=json.loads(target.read_text()) if target.exists() else {}
        target.write_text(json.dumps({**prior,**(body or {})},indent=2))
        return json.loads(target.read_text())
    raise ValueError('Unsupported local record operation')


### Artifact

In [ ]:
def artifact(kind, data, name):
    return api('POST', '/artifacts', {'kind':kind, 'content_b64':base64.b64encode(data).decode(),
        'metadata':{'name':name, 'origin':'synthetic', 'source':'mdCATH molecular dynamics; computational data'}})['artifact_id']

### Atom indices

In [ ]:
def atom_indices(source_keys, target_keys):
    """Map by residue index and atom name; never infer order from array length."""
    from collections import Counter
    counts = Counter(source_keys)
    if any(counts[key] > 1 for key in target_keys):
        raise ValueError('Ambiguous source atom identities')
    lookup = {key:i for i,key in enumerate(source_keys)}
    missing = [key for key in target_keys if key not in lookup]
    if missing:
        raise ValueError(f'Missing model atoms: {missing}')
    return [lookup[key] for key in target_keys]

### Train

In [ ]:
def train(steps=20, track=True):
    if not 1 <= steps <= 200:
        raise ValueError('Choose 1–200 steps for this smoke test')
    os.environ['HF_HOME'] = str(ROOT/'cache')
    import torch
    import numpy as np
    import h5py
    import mdtraj as md
    from huggingface_hub import hf_hub_download
    from bioemu2 import load_model, System, MSA
    from bioemu2.structure.preparation import prepare_structure
    from bioemu2.structure.batch import batch_collate
    if not torch.cuda.is_available():
        raise RuntimeError('Use an NVIDIA CUDA GPU; CPU fallback is disabled')
    torch.manual_seed(42)
    checkpoint = None
    for name in ('config.yaml', 'checkpoint.ckpt'):
        path = hf_hub_download('microsoft/bioemu2', f'checkpoints/bioemu2-preview/{name}', revision=MODEL_REV)
        if name.endswith('.ckpt'): checkpoint = path
    model = load_model(checkpoint_path=checkpoint, device='cuda', precision='bf16-mixed')
    parameter = dict(model.named_parameters())[PARAMETER]
    original = parameter.detach().clone()
    data_manifest = json.loads((ROOT/'data/manifest.json').read_text())
    packages, subsets, selections = [], {}, []
    for domain, count in zip(DOMAINS, (16, 4)):
        entry = next(e for e in data_manifest if e['domain'] == domain)
        file = ROOT/'data'/f'mdcath_dataset_{domain}.h5'
        if hashlib.sha256(file.read_bytes()).hexdigest() != entry['sha256']:
            raise ValueError('mdCATH checksum mismatch')
        with h5py.File(file) as f:
            group = f[domain]
            coords = group['320/0/coords']
            if coords.attrs.get('unit') != 'Angstrom': raise ValueError('Expected Angstrom coordinates')
            pdb = ROOT/'data'/f'{domain}.pdb'
            pdb.write_bytes(group['pdbProteinAtoms'][()])
            topology = md.load_pdb(str(pdb)).topology
            if topology.n_atoms != coords.shape[1]: raise ValueError('Topology/coordinate mismatch')
            sequence = topology.to_fasta()[0]
            system = System.from_description(proteins=[sequence])
            msa = MSA(unpaired_a3m={'A':f'>query\n{sequence}\n'})
            structure = prepare_structure(system, msa=msa, mode='equilibrium', contacts=[])
            indices = atom_indices([(a.residue.index,a.name) for a in topology.atoms],
                                   list(zip(structure.residue_idx.tolist(),structure.atom_names)))
            frames = np.linspace(0,coords.shape[0]-1,count,dtype=int)
            xyz = np.asarray(coords[frames])[:,indices,:]
            xyz -= xyz.mean(axis=1, keepdims=True)
            subsets[domain] = xyz
            selections.append({'domain':domain,'sequence':sequence,'frames':frames.tolist(),
                'temperature_K':320,'replicate':0,'unit':'Angstrom','model_atoms':len(indices),
                'source_sha256':entry['sha256'],'source_url':entry['url']})
        embedding = model.embed(system, msa=msa, mode='equilibrium')
        context = model.compute_static_context(batch_collate([structure]).to('cuda'), embedding, 1)
        target = torch.tensor(xyz, device='cuda')
        # Fixed noise gives a paired baseline/candidate comparison on identical inputs.
        noisy = target + torch.randn_like(target)
        packages.append((context,embedding,target,noisy))
        print(f'Prepared {domain}: {count} frames, {len(indices)} atoms',flush=True)
    with open(checkpoint, 'rb') as stream:
        base_sha256 = hashlib.file_digest(stream, 'sha256').hexdigest()
    config = {'base_checkpoint_sha256':base_sha256,'model_revision':MODEL_REV,'source_revision':SOURCE_REV,'steps':steps,'learning_rate':1e-5,
        'seed':42,'sigma_angstrom':1.0,'trainable_parameter':PARAMETER,'trainable_parameters':parameter.numel(),
        'total_parameters':sum(p.numel() for p in model.parameters()),'gpu_count':1,
        'gpu_name':torch.cuda.get_device_name(0),'device':'cuda','split':selections,
        'method':'Readout-only denoising MSE adaptation; custom experimental recipe',
        'limitations':'Two proteins; fixed noise; query-only MSA; pretraining overlap unknown; no equilibrium sampling validation'}
    buffer=io.BytesIO(); np.savez_compressed(buffer, **subsets)
    dataset_bytes=buffer.getvalue(); digest=hashlib.sha256(dataset_bytes).hexdigest()
    run_id=None
    if track:
        dataset_id=artifact('dataset',dataset_bytes,'mdCATH-small-subset.npz')
        manifest_id=artifact('dataset',json.dumps(config).encode(),'mdCATH-subset-manifest.json')
        run=api('POST','/training/runs',{'kind':'finetune','model_name':'BioEmu-2 · mdCATH readout adaptation',
            'status':'running','dataset_artifact_ids':[dataset_id,manifest_id],'dataset_revision':digest,
            'config':config,'config_hash':hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest(),
            'provenance':{'notebook':'03_bioemu2_mdcath.ipynb','recipe':'bioemu2-readout-adaptation',
                          'model_revision':MODEL_REV,'source_revision':SOURCE_REV}})
        run_id=run['training_run_id'];print('Run:',run_id,flush=True)
    folder=ROOT/'runs'/(run_id or 'smoke-test');folder.mkdir(parents=True,exist_ok=True)
    start=time.monotonic();curve=[]
    gpu_samples=[]; telemetry_stop=threading.Event()
    def sample_gpu():
        while not telemetry_stop.is_set():
            try:
                output=subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,power.draw','--format=csv,noheader,nounits','--id=0'],text=True,timeout=3)
                util,memory,power=[float(v.strip()) for v in output.strip().split(',')]
                gpu_samples.append({'seconds':round(time.monotonic()-start,3),'at':datetime.now(timezone.utc).isoformat(),
                    'utilization_percent':util,'memory_mib':memory,'power_watts':power})
            except (OSError,ValueError,subprocess.SubprocessError):
                pass  # Missing telemetry stays missing; never substitute fabricated readings.
            telemetry_stop.wait(.25)
    telemetry=threading.Thread(target=sample_gpu,daemon=True);telemetry.start()
    torch.cuda.reset_peak_memory_stats()
    def prediction(package,index):
        context,embedding,target,noisy=package
        n = target.shape[1]
        padded = torch.nn.functional.pad(noisy[index:index+1], (0,0,0,(-n) % 32))
        result=model.denoise(context,embedding,padded,torch.ones(1,device='cuda'))
        return result.denoised_coords[:,:n]
    def loss_for(package,index):
        return (prediction(package,index)-package[2][index:index+1]).square().mean()
    def evaluate(package):
        with torch.no_grad():
            return sum(loss_for(package,i).item() for i in range(len(package[2])))/len(package[2])
    wb=None
    try:
        wb=start_run(run_id or ('bioemu-'+__import__('uuid').uuid4().hex[:12]),config,api)
        with torch.no_grad():
            structure_before=prediction(packages[1],0)[0].cpu().tolist()
        before=evaluate(packages[1]);print('Held-out baseline MSE:',before,flush=True)
        parameter.requires_grad_(True)
        optimizer=torch.optim.Adam([parameter],lr=config['learning_rate'])
        for step in range(steps):
            optimizer.zero_grad(set_to_none=True)
            loss=loss_for(packages[0],step % len(packages[0][2]))
            if not torch.isfinite(loss): raise ValueError('Non-finite training loss')
            loss.backward();optimizer.step()
            if wb:wb.log({'train/loss':loss.item(),'train/step':step+1})
            curve.append({'stage':'Readout fine-tuning','epoch':step+1,'loss':loss.item()})
            print(f'Step {step+1}/{steps}: MSE {loss.item():.6f}',flush=True)
            if track:api('PATCH',f'/training/runs/{run_id}',{'metrics':{'curve':curve,'loss':loss.item(),'step':step+1}})
        with torch.no_grad():
            structure_after=prediction(packages[1],0)[0].cpu().tolist()
        after=evaluate(packages[1]);delta=(parameter.detach()-original).norm().item()
        if delta == 0: raise ValueError('Weights did not change')
        ck=folder/'readout.pt';torch.save({'parameter':PARAMETER,'weight':parameter.detach().cpu(),'config':config},ck)
        # Verify the saved adapter restores exactly the trained values.
        restored=torch.load(ck,weights_only=True)['weight'].to('cuda')
        if not torch.equal(restored,parameter.detach()): raise ValueError('Checkpoint round-trip failed')
        telemetry_stop.set();telemetry.join(timeout=4)
        structure_comparison={'domain':DOMAINS[1],'frame':selections[1]['frames'][0],'unit':'Angstrom',
            'view':'C-alpha trace','atom_names':structure.atom_names,'residue_indices':structure.residue_idx.tolist(),
            'reference':packages[1][2][0].cpu().tolist(),'noisy_input':packages[1][3][0].cpu().tolist(),
            'before':structure_before,'after':structure_after,
            'description':'First held-out frame; same noisy input and coordinate system; one-step denoising, not generated equilibrium structures'}
        (folder/'structures.json').write_text(json.dumps(structure_comparison))
        visual={key:structure_comparison[key] for key in ('domain','frame','unit','view')}
        ca=[i for i,name in enumerate(structure.atom_names) if name=='CA']
        visual.update({key:[structure_comparison[key][i] for i in ca] for key in ('reference','before','after')})
        metrics={'structure_comparison':visual,'gpu_samples':gpu_samples,
            'gpu_telemetry_source':'nvidia-smi device readings during this run; includes other GPU activity',
            'peak_torch_memory_gib':torch.cuda.max_memory_allocated()/1024**3,
            'heldout_mse_before' :before,'heldout_mse_after':after,
            'heldout_rmsd_before_angstrom':(3*before)**.5,'heldout_rmsd_after_angstrom':(3*after)**.5,
            'weight_delta_l2':delta,'train_frames':16,'heldout_frames':4,'trainable_parameters':parameter.numel(),
            'gpu_name':config['gpu_name'],'gpu_count':1,'wall_seconds':time.monotonic()-start,'curve':curve,
            'validation':'Paired denoising test on a separate protein; not a scientific benchmark'}
        if wb:finish_run(wb,metrics)
        result={'training_run_id':run_id,'metrics':metrics,'config':config}
        if track:
            structure_id=artifact('dataset',json.dumps(structure_comparison).encode(),'bioemu2-structures.json')
            ck_id=artifact('checkpoint',ck.read_bytes(),'bioemu2-readout.pt')
            log_id=artifact('dataset',json.dumps(result).encode(),'bioemu2-evaluation.json')
            api('PATCH',f'/training/runs/{run_id}',{'status':'succeeded','metrics':metrics,
                'checkpoint_artifact_id':ck_id,'provenance':{'structure_artifact_id':structure_id,'log_artifact_id':log_id,'checkpoint_sha256':hashlib.sha256(ck.read_bytes()).hexdigest()}})
        (folder/'results.json').write_text(json.dumps(result,indent=2))
        print(json.dumps(metrics,indent=2),flush=True)
        print('Results:',folder/'results.json',flush=True)
        return result
    except BaseException as error:
        if wb:wb.finish(exit_code=1)
        telemetry_stop.set();telemetry.join(timeout=4)
        if run_id:api('PATCH',f'/training/runs/{run_id}',{'status':'failed','error':str(error)[-2000:]})
        raise

## Run adaptation

In [ ]:
result=train(steps=20, track=True)

## Before / after · loss · GPU

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
m=result['metrics'];v=m['structure_comparison']
fig=plt.figure(figsize=(15,7))
for position,key,title in [(1,'before','Before adaptation'),(2,'after','After adaptation')]:
    ax=fig.add_subplot(2,2,position,projection='3d')
    ref=np.asarray(v['reference']);xyz=np.asarray(v[key])
    ax.plot(*ref.T,color='lightgray',label='MD reference')
    ax.plot(*xyz.T,color='teal' if key=='after' else 'orange',label=title)
    ax.set_title(title+' · '+v['domain']);ax.legend()
    all_xyz=np.concatenate([np.asarray(v[k]) for k in ('reference','before','after')])
    lo=all_xyz.min(0);hi=all_xyz.max(0)
    ax.set(xlim=(lo[0],hi[0]),ylim=(lo[1],hi[1]),zlim=(lo[2],hi[2]))
ax=fig.add_subplot(2,2,3);ax.plot([r['epoch'] for r in m['curve']],[r['loss'] for r in m['curve']]);ax.set(xlabel='Step',ylabel='Training MSE')
ax=fig.add_subplot(2,2,4);samples=m['gpu_samples']
ax.plot([r['seconds'] for r in samples],[r['utilization_percent'] for r in samples],label='GPU utilization %');ax.set(xlabel='Seconds',ylabel='GPU %',ylim=(0,100));ax.legend()
plt.tight_layout();plt.show()
print('Held-out RMSD (Å):',m['heldout_rmsd_before_angstrom'],'→',m['heldout_rmsd_after_angstrom'])
print('Paired one-step denoising; not equilibrium sampling or a scientific benchmark.')
